# Proyecto Minería de Datos grupo 13 - Hito 1

Integrantes:
- Amaranta Godoy Torres
- Benjamín Silva
- Isidora Reyes M.
- Luciano Tapia T.
- Renato Núñez Díaz

In [2]:
# Primero importamos las librerías a utilizar
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import cross_val_score, KFold
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, make_scorer
import warnings
warnings.filterwarnings('ignore')
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
# Configuramos matplotlib para mostrar gráficos en el notebook
%matplotlib inline

LoadError: ArgumentError: Package pandas not found in current path.
- Run `import Pkg; Pkg.add("pandas")` to install the pandas package.

## Datos a ocupar

Link del data set: https://www.kaggle.com/datasets/ramazanturann/user-animelist-dataset

In [ ]:
# Cargamos los datos
def cargar_datos():
    df_anime = df_anime = pd.read_csv('animes.csv', encoding="UTF-8")
    df_rating = pd.read_csv('ratings.csv', encoding="UTF-8")
    print(f"Animes cargados: {len(df_anime)} registros")
    print(f"Ratings cargados: {len(df_rating)} registros")
    return df_anime, df_rating
df_anime, df_rating = cargar_datos()

## Motivación

El fenómeno del anime es una industria que los últimos años ha crecido de manera significativa, siendo parte ya de la cultura popular tanto en nuestro país como en otras partes del mundo, por eso, el análisis de la valoración de distintos animes podrían ayudar a la industria a proyectar nuevos proyectos para que tengan mejores probabilidades de ser exitosos, estudiar tendencias a través de las décadas o incluso recomendar a personas que quieran adentrarse en este mundo y no saben por dónde empezar.


## Exploración de los datos

En esta sección, se realiza un análisis exploratorio de los datos a utilizar, incluyendo sus carácteristicas, atributos y dimensiones. Además, se realizarán varias visualizaciones que ayudaran a la comprensión del dataset a estudiar.

In [ ]:
#1. Tamaño de los datasets
#Primero con el tamaño de animes
print("Tamaño del dataset de animes", df_anime.shape)
# Ahora con el tamaño de ratings
print("Tamaño del dataset de ratings", df_rating.shape)

In [ ]:
#2. Veamos si existen valores nulos
# Valores nulos en animes
print("Valores nulos en el dataset de animes")
print(df_anime.isnull().sum())
# Valores nulos en ratings
print("Valores nulos en el dataset de ratings")
print(df_rating.isnull().sum())

Se puede ver que en los datos más relevantes de ambos datasets no contienen valores nulos, exceptuando por la columna de títulos alternativos en el dataset de animes.

In [ ]:
#3. Veamos la cantidad de valores únicos en ratings
print("\nUsuarios únicos en ratings:", df_rating['userID'].nunique())
print("Animes únicos en ratings:", df_rating['animeID'].nunique())
print("Número total de calificaciones:", len(df_rating))

Se puede ver que no hay animes repetidos en el dataset, y de ratings podemos
observar que el sitio web cuenta con al menos 1774522 usuarios.

## Visualizaciones

En esta sección realizamos distintas visualizaciones que podrían ser interesantes para nuestro proyecto

In [ ]:
# Distribución de tipos de anime
plt.figure(figsize=(6,4))
df_anime['type'].value_counts().plot(kind='bar')
plt.title("Distribución de tipos de anime")
plt.ylabel("Cantidad")
plt.show()

Se observa que existe una cantidad considerable de animes en cada tipo registrado, destacando que el formato televisivo (TV) es el más frecuente, con cerca de 6.000 títulos.

In [ ]:
# Producción de animes por año
plt.figure(figsize=(12,6))
# Primero filtramos los animes desde 1956 debido a que los otros son más bien valores atípicos.
df_anime['year'].value_counts().sort_index().plot(kind='bar')
plt.title("Número de animes producidos por año")
plt.ylabel("Cantidad")
plt.xticks(rotation=90)
plt.show()

Se aprecia una tendencia coherente con lo señalado en la motivación: la producción de animes experimenta un fuerte aumento a partir de la década de 1990, manteniendo desde entonces un crecimiento sostenido. En particular, el año 2021 destaca como el de mayor número de estrenos dentro de la base de datos analizada.

In [ ]:
 # Calcular score promedio por anime (según ratings de usuarios)
anime_avg_score = df_rating.groupby('animeID')['rating'].mean().reset_index()
# Merge para tener también el título
anime_avg_score = anime_avg_score.merge(df_anime[['animeID','title']], on='animeID', how='left')
# Visualización: distribución del score promedio
plt.figure(figsize=(8,5))
sns.histplot(anime_avg_score['rating'], bins=30, kde=True)
plt.title("Distribución del score promedio por anime (según usuarios)")
plt.xlabel("Score promedio")
plt.ylabel("Cantidad de animes")
plt.show()


Se observa que el puntaje promedio de los animes, cuya escala va de 1 a 10, sigue aproximadamente una distribución normal centrada en torno a un valor de 7.0.

In [ ]:
# Número de animes según cantidad de episodios
# limpiar columna episodes
episodes = pd.to_numeric(df_anime['episodes'].replace('Unknown', np.nan), errors='coerce').dropna().astype(int)
# contar cuántos animes hay por número de episodios
counts = episodes.value_counts().sort_index()
# gráfica de barras horizontal
plt.figure(figsize=(10,6))
sns.barplot(x=counts.values, y=counts.index, color="skyblue")
plt.xlabel("Cantidad de animes")
plt.ylabel("Número de episodios")
plt.title("Número de animes según cantidad de episodios")
# rotar etiquetas y mostrar solo algunas
plt.xticks(rotation=90)   # o 45
plt.show()
plt.show()

El gráfico muestra la distribución exacta del número de animes según su cantidad de episodios. Se observa que los formatos más comunes son los de corta duración: destacan los animes de un solo capítulo, generalmente especiales, OVAs o películas, junto con aquellos de 12 a 13 episodios, correspondientes al estándar de una temporada. También se aprecia un segundo grupo relevante en torno a los 24 a 26 episodios, formato característico de temporadas completas. Más allá de estas concentraciones, la frecuencia disminuye considerablemente, quedando solo algunos casos aislados de producciones más extensas.

In [ ]:
# Ratings por usuario
ratings_per_user = df_rating.groupby('userID').size()
plt.figure(figsize=(8,5))
sns.histplot(ratings_per_user, bins=200)
plt.title("Distribución de cantidad de ratings por usuario")
plt.xlabel("Cantidad de animes puntuados")
plt.ylabel("Número de usuarios")
plt.xlim(0, 1000)
plt.show()

En relación con la cantidad de calificaciones por usuario, se aprecia que la mayoría no supera las 50 evaluaciones registradas. Sin embargo, existe una minoría de usuarios con una actividad considerablemente mayor, alcanzando en algunos casos más de 700 ratings. Esta distribución refleja un comportamiento típico en comunidades en línea, donde una gran proporción de usuarios participa de forma ocasional, mientras que un grupo reducido concentra la mayor parte de la actividad.

## Preguntas de investigación

En esta sección, presentaremos las preguntas de investigación que intentaremos resolver durante el desarrollo del proyecto:

1)   ¿Se puede predecir el género de un anime a partir de su sinopsis y otras variables que se encuentran en la página de MyAnimeList?
2)   ¿Se pueden identificar patrones al agrupar animes mediante clustering? ¿Cómo varía la pureza de los cluster con variables como el score o género del anime?
3)   ¿El score puede predecirse con buena precisión usando solo un subconjunto pequeño de atributos? ¿Qué características son más importantes para explicar/predecir el score de un anime?

## Propuesta Metodológica Experimental

La metodología propuesta para cada pregunta de investigación será la siguiente:

1) **¿Se puede predecir el género de un anime a partir de su sinopsis y otras variables que se encuentran en la página de MyAnimeList?**

    Para responder la primera pregunta, lo primero es trabajar con los datos ya recopilados. En esta etapa, es necesario realizar un preprocesamiento que incluya, por ejemplo, la eliminación de valores nulos en los títulos alternativos y la obtención de la sinopsis mediante web scraping para luego limpiarla. 

    Una vez claras las variables predictoras (sinopsis y otras variables de MyAnimeList) y la variable objetivo (géneros), corresponde dividir los datos en un conjunto de entrenamiento y uno de prueba. También es importante analizar si existe un desbalance de clases, es decir, si algún género tiene una representación significativamente mayor o menor que el resto, ya que eso podría afectar el desempeño del modelo. 

    Finalmente, dado que contamos con información previa (géneros), se trata de un problema de aprendizaje supervisado. En este contexto, el modelo que mejor se ajusta a nuestra pregunta y a este conjunto de datos es Naive Bayes, ya que resulta especialmente eficiente para trabajar con texto. Tras su aplicación, corresponde realizar la validación adecuada para medir su desempeño y comprobar si logra predecir correctamente los géneros de los animes.

2) **¿Se pueden identificar patrones al agrupar animes mediante clustering? ¿Cómo varía la pureza de los cluster con variables como el score o género del anime?**

    Para responder la segunda pregunta, primero se realizará un preprocesamiento de los datos separando las variables en numéricas, categóricas y de texto, aplicando normalización y codificación apropiada, además de eliminar outliers en caso de detectarse. 

    Luego, se aplicarán tres algoritmos de clustering de distinta naturaleza —K-means, clustering jerárquico aglomerativo y DBSCAN— con el fin de comparar los grupos que generan. 

    La evaluación se llevará a cabo con métricas internas (como Silhouette y Davies-Bouldin) y externas, midiendo la pureza de los clústeres respecto al género y categorías del score, así como su variación al modificar parámetros como el número de clústeres o los valores de eps y minPts. El uso de distintos algoritmos permitirá observar cómo varían los patrones descubiertos y obtener una visión más completa de la estructura de los datos.


3)   **¿El score puede predecirse con buena precisión usando solo un subconjunto pequeño de atributos? ¿Qué características son más importantes para explicar/predecir el score de un anime?**

        Para responder la tercera pregunta, una vez aplicado el pre procesamiento adicional necesario, se utilizarán modelos de regresión supervisada. 
    
        En primer lugar se emplearán modelos lineales: Regresión lineal, Lasso y Ridge, que evalúan directamente cuánto influye cada feature en la variable objetivo, en este caso, el Score Además, de Lasso realiza una selección automática de las variables relevantes.Posteriormente, se entrenarán modelos más robustos basados en árboles de decisión mediante métodos de ensamble como Random forest y Boosting, con el fin de capturar relaciones no lineales y evaluar si aportan mejoras significativas en el desempeño. 
    
        La selección de características en los modelos lineales se realizará con Lasso y para los modelos de ensamble se hará mediante Permutation Importance, técnica que consiste en desordenar los valores de una variable y medir cómo afectan en el desempeño del modelo, de esta forma se identifican aquellas variables más importantes. Una vez obtenidas las variables relevantes, se comparará el desempeño de los modelos entrenados con  todas las características frente a los entrenados con un conjunto reducido.

        Para la evaluación de los resultados se utilizará validación cruzada de cinco folds, cuatro de entrenamiento y uno de prueba en cada iteración. Para ello se utilizarán las métricas de error medio absoluto, error cuadrático medio y el coeficiente R^2. Los resultados se compararán con un modelo baseline que predice simplemente la media  del Score de todos los animes, y se analizará como varía el rendimiento de los modelos al usar todas las variables frente a un con un conjunto reducido de las más relevantes.



## Resultados Preliminares



Para el experimento preliminar, se realizo una primera iteración del modelo para la pregunta 3

In [ ]:
# Primero, se hace una limpieza de los datos para preparar su uso:

# Limpieza de datos
df = df_anime.copy()
df = df[df['score'] != '?']
df['score'] = df['score'].astype(float)
df['episodes'] = pd.to_numeric(df['episodes'], errors='coerce')
df['year'] = pd.to_numeric(df['year'], errors='coerce')
# Quitar filas con NA en las columnas importantes
df = df.dropna(subset=['episodes', 'year', 'score', 'type', 'genres'])

# Se define la variable predictiva y (correspondiente al score), y el primer set de variables X que se usará para realizar la predicción (type, genres, episodes, year):

y = df['score']
X = df[['type', 'genres', 'episodes', 'year']].copy()
# Como type y genres no son variables numéricas, se procesan para crear columnas binarias para cada dato único
preprocessor = ColumnTransformer(
    transformers=[
        ('type_enc', OneHotEncoder(handle_unknown='ignore'), ['type']),
        ('genres_enc', OneHotEncoder(handle_unknown='ignore'), ['genres']),],
    remainder='passthrough' 
)

# Despues, se definen las métricas a utilizar en el experimento. En este caso, utilizamos el MAE, el RMSE y el $R^2$

scoring = {
    "MAE": make_scorer(mean_absolute_error, greater_is_better=False),
    "RMSE": make_scorer(lambda y_true, y_pred: np.sqrt(mean_squared_error(y_true, y_pred)), greater_is_better=False),
    "R2": make_scorer(r2_score)
}

# Luego se realiza la partición de los datos usando KFold y preparamos la baseline

kf = KFold(n_splits=5, shuffle=True, random_state=42) # Separamos los datos en 5 particiones
y_baseline = np.full_like(y, fill_value=y.mean())
mae_base = mean_absolute_error(y, y_baseline)
rmse_base = np.sqrt(mean_squared_error(y, y_baseline))
r2_base = r2_score(y, y_baseline)
print("=== Baseline (media del score) ===")
print(f"MAE:  {mae_base:.3f}")
print(f"RMSE: {rmse_base:.3f}")
print(f"R²:   {r2_base:.3f}\n")

# Después, se implementan los modelos descritos en la metodología y se recopilan los resultados de cada uno 

modelos = {
    "LinearRegression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "Lasso": Lasso(alpha=0.01, max_iter=10000),
    "RandomForest": RandomForestRegressor(n_estimators=100, random_state=42),
    "GradientBoosting": GradientBoostingRegressor(n_estimators=200, learning_rate=0.1, random_state=42)
}
resultados = []
for nombre, modelo in modelos.items():
    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('model', modelo)
    ])
    mae = -cross_val_score(pipe, X, y, cv=kf, scoring=scoring["MAE"]).mean()
    rmse = -cross_val_score(pipe, X, y, cv=kf, scoring=scoring["RMSE"]).mean()
    r2 = cross_val_score(pipe, X, y, cv=kf, scoring=scoring["R2"]).mean()
    resultados.append({
        "Modelo": nombre,
        "MAE": mae,
        "RMSE": rmse,
        "R²": r2
    })
# Creación de df con los resultados
df_resultados = pd.DataFrame(resultados)
print("\n=== Resultados comparativos ===")
print(df_resultados)


### Interpretación de los resultados

Luego de aplicar los modelos se obtienen los resultados. 

Se observa que, para todos los modelos, los valores de MAE y RMSE disminuyen en comparación con la línea base, mostrando una mejora cercana al 20%. No obstante, al analizar el coeficiente $R^2$, acotado entre menos infinito y 1, los resultados obtenidos no se acercan lo suficiente a 1. Esto indica que los modelos solo logran explicar aproximadamente un 34% de la variabilidad del puntaje.

Este comportamiento podría deberse a que la variable score está influenciada por factores subjetivos que no se encuentran presentes en el conjunto de datos, como la popularidad de los animes. Por ello, el siguiente paso será incorporar información adicional obtenida desde la URL mediante técnicas de web scraping, con el fin de evaluar si estos nuevos atributos contribuyen a explicar la variabilidad del score.

En caso de que no se observe una mejora significativa en las métricas, podrá concluirse que, si bien ciertas características estructurales del anime tienen cierta capacidad predictiva, estas no resultan suficientes para modelar un fenómeno complejo y multifactorial como lo es la calificación de un anime.

Contribución de cada integrante

Hay varias partes del hito que fueron desarrolladas en conjunto en sesiones de trabajo grupal: búsqueda de datasets, elaboración de preguntas de investigación, elaboración del código del experimento preliminar y elaboración del informe y la presentación.

Luego, el aporte individual de cada integrante es el siguiente:

- Amaranta Godoy Torres: Visualizaciones y Motivación.
- Benjamín Silva: Inicio de la exploración de los datos e interpretación de los resultados.
- Isidora Reyes M.: Propuesta Metodológica Experimental de la tercera pregunta de investigación.
- Luciano Tapia T.: Propuesta Metodológica Experimental de la primera pregunta de investigación.
- Renato Núñez Díaz: Propuesta Metodológica Experimental de la segunda pregunta de investigación.